In [1]:
# Load libraries used for ingestion, cleaning, and filesystem operations.
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Define project paths and load each source table with encoding fallback.
RAW_DIR = Path("../data/raw")
CLEAN_DIR = Path("../data/cleaned")
REPORT_DIR = Path("../reports")

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def read_csv_safely(path):
    for encoding in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue

    raise ValueError(f"Could not read file: {path}")


customers = read_csv_safely(RAW_DIR / "Customers.csv")
products = read_csv_safely(RAW_DIR / "Products.csv")
sales = read_csv_safely(RAW_DIR / "Sales.csv")
stores = read_csv_safely(RAW_DIR / "Stores.csv")
exchange_rates = read_csv_safely(RAW_DIR / "Exchange_Rates.csv")
data_dictionary = read_csv_safely(RAW_DIR / "Data_Dictionary.csv")

In [3]:
# Normalize column names and text values before applying type-specific cleaning.
tables = {
    "customers": customers,
    "products": products,
    "sales": sales,
    "stores": stores,
    "exchange_rates": exchange_rates,
    "data_dictionary": data_dictionary
}

for name, df in tables.items():
    df.columns = df.columns.str.strip()

    # Remove leading/trailing whitespace from text values
    text_columns = df.select_dtypes(include=["object", "string"]).columns

    for column in text_columns:
        df[column] = df[column].str.strip()

        # Convert empty strings into proper missing values
        df[column] = df[column].replace("", np.nan)

In [4]:
# Inspect the product table before transforming monetary fields.
products.head()

,ProductKey,Product Name,Brand,Color,Unit Cost USD,Unit Price USD,SubcategoryKey,Subcategory,CategoryKey,Category
0,1,Contoso 512MB MP3 Player E51 Silver,Contoso,Silver,$6.62,$12.99,101,MP4&MP3,1,Audio
1,2,Contoso 512MB MP3 Player E51 Blue,Contoso,Blue,$6.62,$12.99,101,MP4&MP3,1,Audio
2,3,Contoso 1G MP3 Player E100 White,Contoso,White,$7.40,$14.52,101,MP4&MP3,1,Audio
3,4,Contoso 2G MP3 Player E200 Silver,Contoso,Silver,$11.00,$21.57,101,MP4&MP3,1,Audio
4,5,Contoso 2G MP3 Player E200 Red,Contoso,Red,$11.00,$21.57,101,MP4&MP3,1,Audio


In [5]:
# Convert currency-formatted product values into numeric USD measures.
money_columns = ["Unit Cost USD", "Unit Price USD"]

for column in money_columns:
    products[column] = pd.to_numeric(
        products[column].astype("string")
        .str.replace("$", "",regex=False)
        .str.replace(",", "",regex=False)
        .str.strip(),
        errors="coerce"
    )

In [16]:
# Verify that monetary conversion produced usable numeric values.
print(products[money_columns].dtypes)
print(products[money_columns].isna().sum())
products[money_columns].info()

Unit Cost USD     Float64
Unit Price USD    Float64
dtype: object
Unit Cost USD     0
Unit Price USD    0
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 2517 entries, 0 to 2516
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unit Cost USD   2517 non-null   Float64
 1   Unit Price USD  2517 non-null   Float64
dtypes: Float64(2)
memory usage: 44.4 KB


In [6]:
# Parse date fields so date validation and time-based analysis are reliable.
sales["Order Date"] = pd.to_datetime(
    sales["Order Date"], errors="coerce"
)

sales["Delivery Date"] = pd.to_datetime(
    sales["Delivery Date"], errors="coerce"
)

stores["Open Date"] = pd.to_datetime(
    stores["Open Date"], errors="coerce"
)

exchange_rates["Date"] = pd.to_datetime(
    exchange_rates["Date"], errors="coerce"
)

### Handle missing values based on business meaning


In [7]:
# Inspect the known missing customer state-code records before deciding how to retain them.
print(
    customers[customers["State Code"].isna()][
        ["CustomerKey", "City", "State", "Country", "State Code"]
    ]
)

      CustomerKey               City   State Country State Code
7158       729681            Polvica  Napoli   Italy        NaN
7159       732289          Varcaturo  Napoli   Italy        NaN
7160       742042          Casaferro  Napoli   Italy        NaN
7161       742886           Terzigno  Napoli   Italy        NaN
7162       743343  Pomigliano D'Arco  Napoli   Italy        NaN
7163       759705            Casilli  Napoli   Italy        NaN
7164       765589       Pompei Scavi  Napoli   Italy        NaN
7165       766410             Licola  Napoli   Italy        NaN
7166       781667             Napoli  Napoli   Italy        NaN
7167       789177      Mariglianella  Napoli   Italy        NaN


In [8]:
# Inspect missing store-size values, including the online channel.
print(
    stores[stores["Square Meters"].isna()][
        ["StoreKey", "Country", "State", "Square Meters"]
    ]
)

    StoreKey Country   State  Square Meters
66         0  Online  Online            NaN


### Checkiing for Duplicates


In [9]:
# Check for exact duplicate rows across all source tables.
for name, df in tables.items():
    print(name, "exact duplicate rows:", df.duplicated().sum())

customers exact duplicate rows: 0
products exact duplicate rows: 0
sales exact duplicate rows: 0
stores exact duplicate rows: 0
exchange_rates exact duplicate rows: 0
data_dictionary exact duplicate rows: 0


In [10]:
# Check whether an order-line identifier is duplicated within Sales.
duplicate_lines = sales.duplicated(
    subset=["Order Number", "Line Item"],
    keep="first"
)

print("Rows involved in duplicate order lines:", duplicate_lines.sum())

Rows involved in duplicate order lines: 0


In [13]:
# Verify that dimension keys are unique before using them in joins.
print("Duplicate customer keys:", customers["CustomerKey"].duplicated().sum())
print("Duplicate product keys:", products["ProductKey"].duplicated().sum())
print("Duplicate store keys:", stores["StoreKey"].duplicated().sum())

Duplicate customer keys: 0
Duplicate product keys: 0
Duplicate store keys: 0


In [17]:
# Confirm that every sales foreign key maps to a dimension record.
print(
    "Unknown customer keys:",
    (~sales["CustomerKey"].isin(customers["CustomerKey"])).sum()
)

print(
    "Unknown product keys:",
    (~sales["ProductKey"].isin(products["ProductKey"])).sum()
)

print(
    "Unknown store keys:",
    (~sales["StoreKey"].isin(stores["StoreKey"])).sum()
)

Unknown customer keys: 0
Unknown product keys: 0
Unknown store keys: 0


In [18]:
# Check quantity quality before calculating sales metrics.
print(sales["Quantity"].dtype)
print(sales["Quantity"].describe())

print("Missing quantities:", sales["Quantity"].isna().sum())
print("Zero or negative quantities:", (sales["Quantity"] <= 0).sum())

int64
count    62884.000000
mean         3.144790
std          2.256371
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         10.000000
Name: Quantity, dtype: float64
Missing quantities: 0
Zero or negative quantities: 0


In [19]:
# Review the currencies present in Sales, including missing currency codes.
print(sales["Currency Code"].value_counts(dropna=False))
# dropna=False is used to include 

Currency Code
USD    33767
EUR    12621
GBP     8140
CAD     5415
AUD     2941
Name: count, dtype: int64


In [20]:
# Check whether each sales date and currency has a corresponding exchange-rate record.
rate_keys = exchange_rates[
    ["Date", "Currency"]
].drop_duplicates()

sales_rate_keys = sales[
    ["Order Date", "Currency Code"]
].drop_duplicates()

coverage = sales_rate_keys.merge(
    rate_keys,
    left_on=["Order Date", "Currency Code"],
    right_on=["Date", "Currency"],
    how="left",
    indicator=True
)

print(coverage["_merge"].value_counts())

_merge
both          6004
left_only        0
right_only       0
Name: count, dtype: int64


In [21]:
# Finalize numeric types and refresh the table registry used by reporting steps.
# Ensure numeric columns have the correct types
sales["Quantity"] = pd.to_numeric(
    sales["Quantity"], errors="coerce"
)

stores["Square Meters"] = pd.to_numeric(
    stores["Square Meters"], errors="coerce"
)

exchange_rates["Exchange"] = pd.to_numeric(
    exchange_rates["Exchange"], errors="coerce"
)

# Refresh table references after cleaning
tables = {
    "Customers": customers,
    "Products": products,
    "Sales": sales,
    "Stores": stores,
    "Exchange_Rates": exchange_rates,
    "Data_Dictionary": data_dictionary
}

In [22]:
# Create a compact data-quality report for each cleaned table.
quality_report = []

for name, df in tables.items():
    quality_report.append({
        "Table": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Cells": int(df.isna().sum().sum()),
        "Exact Duplicate Rows": int(df.duplicated().sum())
    })

quality_report = pd.DataFrame(quality_report)

display(quality_report)

quality_report.to_csv(
    REPORT_DIR / "data_quality_report.csv",
    index=False
)

,Table,Rows,Columns,Missing Cells,Exact Duplicate Rows
0,Customers,15266,10,10,0
1,Products,2517,10,0,0
2,Sales,62884,9,49719,0
3,Stores,67,5,1,0
4,Exchange_Rates,11215,3,0,0
5,Data_Dictionary,37,3,0,0


In [26]:
# Write cleaned tables to the data/cleaned directory for downstream analysis.
for name, df in tables.items():
    filename = name + ".csv"

    df.to_csv(
        CLEAN_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.


In [24]:
# Recalculate core metrics from the exported files to confirm the cleaning output is usable.
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/cleaned")

sales = pd.read_csv(DATA_DIR / "Sales.csv")
products = pd.read_csv(DATA_DIR / "Products.csv")

# Keep the original sales row count
original_rows = len(sales)

# Check that product keys are unique
assert products["ProductKey"].is_unique

# Add product price and cost to each sales line
check = sales.merge(
    products[["ProductKey", "Unit Price USD", "Unit Cost USD"]],
    on="ProductKey",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Confirm that the merge preserved all sales lines
assert len(check) == original_rows

# Check that every sales line found a product
print(check["_merge"].value_counts())

# Calculate the core metrics
check["Revenue USD"] = (
    check["Quantity"] * check["Unit Price USD"]
)

check["COGS USD"] = (
    check["Quantity"] * check["Unit Cost USD"]
)

check["Gross Profit USD"] = (
    check["Revenue USD"] - check["COGS USD"]
)

revenue = check["Revenue USD"].sum()
cogs = check["COGS USD"].sum()
gross_profit = check["Gross Profit USD"].sum()
orders = check["Order Number"].nunique()
units = check["Quantity"].sum()

print(f"Revenue: ${revenue:,.2f}")
print(f"COGS: ${cogs:,.2f}")
print(f"Gross profit: ${gross_profit:,.2f}")
print(f"Gross margin: {gross_profit / revenue:.2%}" if revenue else "Gross margin: N/A")
print(f"Distinct orders: {orders:,}")
print(f"Units sold: {units:,}")
print(f"Average order value: ${revenue / orders:,.2f}" if orders else "AOV: N/A")

_merge
both          62884
left_only         0
right_only        0
Name: count, dtype: int64
Revenue: $55,755,479.59
COGS: $23,092,791.21
Gross profit: $32,662,688.38
Gross margin: 58.58%
Distinct orders: 26,326
Units sold: 197,757
Average order value: $2,117.89
